In [ ]:
# ============================================================
# LABEL EFFICIENCY ANALYSIS | DentAttn-MSFNet | best fold
# Per-class metrics + resumable (Kaggle kernel reset safe)
# ============================================================
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
warnings.filterwarnings("ignore")

# -------------------- SETTINGS --------------------
FOLD       = 0
FRACTIONS  = [0.10, 0.25, 0.50, 0.75, 1.00]   
SEEDS      = [42]                             
LE_DIR     = "/kaggle/working/label_efficiency_fold0"
os.makedirs(LE_DIR, exist_ok=True)

# -------------------- HELPERS --------------------
def stratified_subset(train_idx, frac, seed):
    """Class-wise nested subset (same seed -> 10% ⊂ 25% ⊂ 50% ...)"""
    train_idx = np.asarray(train_idx)
    rng = np.random.RandomState(seed)
    out = []
    for c in range(len(CLASSES)):
        ids = rng.permutation(train_idx[y[train_idx] == c])
        n_take = min(len(ids), max(int(round(frac * len(ids))), 2))
        out.append(ids[:n_take])
    return np.sort(np.concatenate(out))

def make_loader_le(idx, tf, shuffle):
    drop = shuffle and (len(idx) > CFG["batch_size"])  
    return DataLoader(DentalDS(X, y, idx, tf), batch_size=CFG["batch_size"],
                      shuffle=shuffle, num_workers=0, pin_memory=True, drop_last=drop)

def train_subset(sub_idx, val_idx, tag):
    tr_loader = make_loader_le(sub_idx, train_tf, True)
    va_loader = make_loader_le(val_idx, eval_tf, False)

    cnt = np.bincount(y[sub_idx], minlength=len(CLASSES))
    w = torch.tensor(len(sub_idx) / (len(CLASSES) * cnt + 1e-6), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["label_smoothing"])

    model = build_model()
    head_params = (list(model.fems.parameters()) + list(model.attns.parameters()) +
                   list(model.fc.parameters()))
    if CFG["use_bifpn"]:
        head_params += list(model.fusion.parameters())

    opt = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": CFG["lr"]},
        {"params": head_params, "lr": CFG["lr"] * 3},
    ], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=CFG["warmup_epochs"]),
        torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(CFG["epochs"] - CFG["warmup_epochs"], 1))
    ], milestones=[CFG["warmup_epochs"]])
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_f1, hist = -1.0, []
    ckpt = os.path.join(LE_DIR, f"best_{tag}.pth")

    for ep in range(CFG["epochs"]):
        model.train()
        run_loss, n = 0.0, 0
        for xb, yb in tqdm(tr_loader, desc=f"{tag} | Ep {ep+1}/{CFG['epochs']}", leave=False):
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(xb).float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            run_loss += loss.item() * len(yb); n += len(yb)
        sched.step()

        _, labels, preds = evaluate(model, va_loader)
        val_acc = accuracy_score(labels, preds)
        val_f1  = f1_score(labels, preds, average="macro")
        hist.append({"epoch": ep+1, "train_loss": run_loss/n, "val_acc": val_acc, "val_f1": val_f1})
        flag = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), ckpt)
            flag = " ✅"
        if (ep + 1) % 5 == 0 or ep == 0 or flag:
            print(f"{tag} | Ep {ep+1:02d} | loss {run_loss/n:.4f} | val_acc {val_acc:.4f} | macroF1 {val_f1:.4f}{flag}")

    model.load_state_dict(torch.load(ckpt, map_location=device))
    return model, pd.DataFrame(hist)

# -------------------- RUN (resumable) --------------------
fd = folds[FOLD]
te_loader = make_loader_le(fd["test"], eval_tf, False)
summary_path = os.path.join(LE_DIR, "le_summary.csv")

for seed in SEEDS:
    for frac in FRACTIONS:
        tag = f"f{FOLD}_p{int(frac*100):03d}_s{seed}"
        done_flag = os.path.join(LE_DIR, f"perclass_{tag}.csv")
        if os.path.exists(done_flag):
            print(f"⏭️  {tag} already done, skipping")
            continue

        sub_idx = stratified_subset(fd["train"], frac, seed)
        cnts = np.bincount(y[sub_idx], minlength=len(CLASSES))
        print(f"\n{'='*70}\n🚀 {tag} | train samples = {len(sub_idx)} | per-class = {dict(zip(CLASSES, cnts))}\n{'='*70}")

        t0 = time.time()
        model, hist = train_subset(sub_idx, fd["val"], tag)
        probs, labels, preds = evaluate(model, te_loader)

        acc = accuracy_score(labels, preds)
        mf1 = f1_score(labels, preds, average="macro")
        wf1 = f1_score(labels, preds, average="weighted")

        pc = per_class_metrics(labels, preds, CLASSES)
        pc.insert(0, "Label Fraction (%)", int(frac * 100))
        pc.insert(1, "Seed", seed)
        pc.insert(2, "N Train", len(sub_idx))
        print(f"\n----- {tag} Per-Class Metrics -----")
        print(pc.to_string(index=False))
        print(f"Test → Acc {acc:.4f} | Macro-F1 {mf1:.4f} | Weighted-F1 {wf1:.4f} | {(time.time()-t0)/60:.1f} min")

        hist.to_csv(os.path.join(LE_DIR, f"history_{tag}.csv"), index=False)
        pd.DataFrame(confusion_matrix(labels, preds), index=CLASSES, columns=CLASSES)\
          .to_csv(os.path.join(LE_DIR, f"cm_{tag}.csv"))
        pd.DataFrame([{"fraction": int(frac*100), "seed": seed, "n_train": len(sub_idx),
                       "accuracy": acc, "macro_f1": mf1, "weighted_f1": wf1}])\
          .to_csv(os.path.join(LE_DIR, f"summary_{tag}.csv"), index=False)
        pc.to_csv(done_flag, index=False)     # last write = "done" marker

        del model; torch.cuda.empty_cache()

# -------------------- AGGREGATE --------------------
pc_all = pd.concat([pd.read_csv(os.path.join(LE_DIR, f)) for f in sorted(os.listdir(LE_DIR))
                    if f.startswith("perclass_")], ignore_index=True)
sm_all = pd.concat([pd.read_csv(os.path.join(LE_DIR, f)) for f in sorted(os.listdir(LE_DIR))
                    if f.startswith("summary_f")], ignore_index=True)
pc_all.to_csv(os.path.join(LE_DIR, "le_per_class_all.csv"), index=False)

sm_agg = sm_all.groupby("fraction")[["n_train", "accuracy", "macro_f1", "weighted_f1"]].agg(["mean", "std"]).round(4)
sm_agg.to_csv(summary_path)
print("\n" + "="*70 + "\nLABEL EFFICIENCY SUMMARY (Fold %d)\n" % FOLD + "="*70)
print(sm_agg.to_string())

# Per-class tables (mean over seeds): rows = class, cols = fraction
metric_cols = ["Accuracy (%)", "TPR (%)", "Precision (%)", "F1 Score (%)"]
for m in metric_cols:
    piv = pc_all.pivot_table(index="Class", columns="Label Fraction (%)", values=m, aggfunc="mean").round(2)
    piv = piv.reindex(CLASSES)
    piv.to_csv(os.path.join(LE_DIR, f"le_pivot_{m.split(' (')[0].replace(' ', '_')}.csv"))
    print(f"\n--- {m} (rows=class, cols=label %) ---")
    print(piv.to_string())

# -------------------- PLOTS --------------------
fracs = sorted(pc_all["Label Fraction (%)"].unique())
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (a) overall
ax = axes[0, 0]
g = sm_all.groupby("fraction")
for key, lbl, c in [("accuracy", "Accuracy", "tab:green"), ("macro_f1", "Macro-F1", "tab:orange"),
                    ("weighted_f1", "Weighted-F1", "tab:blue")]:
    mu, sd = g[key].mean() * 100, g[key].std().fillna(0) * 100
    ax.plot(mu.index, mu.values, "o-", lw=2, color=c, label=lbl)
    ax.fill_between(mu.index, mu - sd, mu + sd, alpha=0.15, color=c)
ax.set_xlabel("Labeled training data (%)"); ax.set_ylabel("Test score (%)")
ax.set_title("Overall Performance vs Label Fraction"); ax.set_xticks(fracs)
ax.legend(); ax.grid(True, alpha=0.3)

# (b) per-class F1
ax = axes[0, 1]
for cname in CLASSES:
    d = pc_all[pc_all["Class"] == cname].groupby("Label Fraction (%)")["F1 Score (%)"].mean()
    ax.plot(d.index, d.values, "o-", lw=2, label=cname)
ax.set_xlabel("Labeled training data (%)"); ax.set_ylabel("F1 (%)")
ax.set_title("Per-Class F1 vs Label Fraction"); ax.set_xticks(fracs)
ax.legend(); ax.grid(True, alpha=0.3)

# (c) per-class recall (TPR)
ax = axes[1, 0]
for cname in CLASSES:
    d = pc_all[pc_all["Class"] == cname].groupby("Label Fraction (%)")["TPR (%)"].mean()
    ax.plot(d.index, d.values, "s-", lw=2, label=cname)
ax.set_xlabel("Labeled training data (%)"); ax.set_ylabel("Recall / TPR (%)")
ax.set_title("Per-Class Recall vs Label Fraction"); ax.set_xticks(fracs)
ax.legend(); ax.grid(True, alpha=0.3)

# (d) heatmap per-class F1
ax = axes[1, 1]
piv = pc_all.pivot_table(index="Class", columns="Label Fraction (%)", values="F1 Score (%)", aggfunc="mean").reindex(CLASSES)
sns.heatmap(piv, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={"label": "F1 (%)"}, ax=ax)
ax.set_title("Per-Class F1 Heatmap"); ax.set_xlabel("Labeled training data (%)")

plt.suptitle(f"DentAttn-MSFNet | Label Efficiency Analysis (Fold {FOLD})",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(LE_DIR, "label_efficiency.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✅ Done! Saved in: {LE_DIR}")
print("   - le_summary.csv, le_per_class_all.csv, le_pivot_*.csv")
print("   - perclass_/history_/cm_/summary_ per run, label_efficiency.png (300 DPI)")